# CVR DATA API. Only 100-200 requests pr. day

In [ ]:
import os
import json
import random
import pandas as pd
import requests
import leafmap
import json
import time

In [ ]:


# Load the CVR JSON file
cvr_path = os.path.join("..", "data", "raw", "cvr_raw",
                        "CVR_V1_Virksomhed_TotalDownload_json_Current_317.json")
with open(cvr_path, "r", encoding="utf-8") as f:
    cvr_data = json.load(f)

# Extract all CVR numbers
all_cvr_numbers = [entry["CVRNummer"] for entry in cvr_data if "CVRNummer" in entry]
print(f"Total CVR numbers loaded: {len(all_cvr_numbers)}")


def get_random_cvr_numbers(cvr_list, n=10):
    """Return a list of n random CVR numbers from cvr_list."""
    n = min(n, len(cvr_list))
    return random.sample(cvr_list, n)


# Set the number of random CVR numbers you want
num_random = 100
random_cvr_numbers = get_random_cvr_numbers(all_cvr_numbers, n=num_random)
print(f"Selected {len(random_cvr_numbers)} random CVR numbers")
print(random_cvr_numbers[:10])  # preview first 10

Total CVR numbers loaded: 865485
Selected 100 random CVR numbers
[29631956, 33758626, 45865312, 29536198, 33540809, 44802961, 33289286, 45153894, 34738947, 46230043]


In [ ]:
cvr_list = [16639435, 21275913]

#random_cvr_numbers 

In [ ]:
# === Full pipeline: CVR list → production units → DAWA geocoding ===
# Supports resuming from a checkpoint file so you can continue across quota resets.

CHECKPOINT_PATH = os.path.join("..", "data", "interim", "cvr_pipeline_checkpoint.json")

def cvrapi_lookup(cvr, country='dk'):
    """Fetch company + production units from cvrapi.dk for a single CVR number."""
    import urllib.request as req
    import contextlib
    r = req.Request(
        url='http://cvrapi.dk/api?search=%d&country=%s' % (int(cvr), country),
        headers={'User-Agent': 'CVR_gentrification_model'})
    with contextlib.closing(req.urlopen(r)) as resp:
        return json.loads(resp.read())

def get_adgangs_uuid(address_str):
    """Look up an address string via DAWA datavask and return the adgangsadresseid."""
    if not address_str or pd.isna(address_str):
        return None
    try:
        resp = requests.get(
            "https://api.dataforsyningen.dk/datavask/adresser",
            params={"betegnelse": address_str})
        if resp.status_code != 200:
            return None
        resultater = resp.json().get("resultater", [])
        if resultater:
            return resultater[0].get("adresse", {}).get("adgangsadresseid")
    except Exception as e:
        print(f"  Datavask fejl for '{address_str}': {e}")
    return None

def get_coords_from_uuid(uuid):
    """Fetch lon/lat from DAWA adgangsadresser endpoint."""
    if not uuid:
        return None, None
    try:
        resp = requests.get(
            f"https://api.dataforsyningen.dk/adgangsadresser/{uuid}",
            params={"format": "geojson", "struktur": "flad", "geometri": "adgangspunkt"})
        if resp.status_code != 200:
            return None, None
        coords = resp.json().get("geometry", {}).get("coordinates", [])
        if len(coords) >= 2:
            return coords[0], coords[1]  # lon, lat
    except Exception as e:
        print(f"  Coord fejl for UUID {uuid}: {e}")
    return None, None

def save_checkpoint(rows, done_cvrs):
    """Save current results + processed CVR set to disk."""
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump({"rows": rows, "done_cvrs": list(done_cvrs)}, f, ensure_ascii=False)

def load_checkpoint():
    """Load previous checkpoint if it exists."""
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            cp = json.load(f)
        print(f"Resuming from checkpoint: {len(cp['done_cvrs'])} CVRs already processed, {len(cp['rows'])} rows collected.")
        return cp["rows"], set(cp["done_cvrs"])
    return [], set()


# Load any previous progress
all_rows, done_cvrs = load_checkpoint()
remaining = [c for c in cvr_list if c not in done_cvrs]
print(f"Total: {len(cvr_list)} | Already done: {len(done_cvrs)} | Remaining: {len(remaining)}")

for i, cvr in enumerate(remaining):
    print(f"\n[{len(done_cvrs)+i+1}/{len(cvr_list)}] --- CVR {cvr} ---")
    try:
        api_data = cvrapi_lookup(cvr)
    except Exception as e:
        print(f"  Kunne ikke hente CVR {cvr}: {e}")
        continue

    # Check for API errors (e.g. QUOTA_EXCEEDED)
    if "error" in api_data:
        print(f"  API fejl: {api_data.get('error')} – {api_data.get('message', '')}")
        if api_data.get("error") == "QUOTA_EXCEEDED":
            print(f"Quota exceeded – stopping. {len(done_cvrs)} CVRs saved to checkpoint.")
            print("Re-run this cell later to resume from where you left off.")
            save_checkpoint(all_rows, done_cvrs)
            break
        continue

    company_name = api_data.get("name", "")
    units = api_data.get("productionunits") or []

    if not units:
        # Fallback: use the company-level address when no production units exist
        print("  Ingen produktionsenheder – bruger virksomhedsadresse.")
        addr_str = (
            (api_data.get("address") or "") + ", " +
            str(api_data.get("zipcode") or "") + " " +
            (api_data.get("city") or "")
        )
        uuid = get_adgangs_uuid(addr_str)
        lon, lat = get_coords_from_uuid(uuid)
        print(f"  Virksomhed: {addr_str.strip(', ')}  → UUID={uuid}  ({lon}, {lat})")

        all_rows.append({
            "cvr": cvr,
            "company": company_name,
            "pno": None,
            "name": company_name,
            "address": api_data.get("address"),
            "zipcode": api_data.get("zipcode"),
            "city": api_data.get("city"),
            "startdate": api_data.get("startdate"),
            "enddate": api_data.get("enddate"),
            "employees": api_data.get("employees"),
            "industrycode": api_data.get("industrycode"),
            "industrydesc": api_data.get("industrydesc"),
            "adgangs_uuid": uuid,
            "lon": lon,
            "lat": lat,
        })
    else:
        for idx, pu in enumerate(units):
            pno = pu.get("pno")
            addr_str = (
                (pu.get("address") or "") + ", " +
                str(pu.get("zipcode") or "") + " " +
                (pu.get("city") or "")
            )

            uuid = get_adgangs_uuid(addr_str)
            lon, lat = get_coords_from_uuid(uuid)
            print(f"  [{idx+1}/{len(units)}] pno={pno}  {addr_str.strip(', ')}  → UUID={uuid}  ({lon}, {lat})")

            all_rows.append({
                "cvr": cvr,
                "company": company_name,
                "pno": pno,
                "name": pu.get("name"),
                "address": pu.get("address"),
                "zipcode": pu.get("zipcode"),
                "city": pu.get("city"),
                "startdate": pu.get("startdate"),
                "enddate": pu.get("enddate"),
                "employees": pu.get("employees"),
                "industrycode": pu.get("industrycode"),
                "industrydesc": pu.get("industrydesc"),
                "adgangs_uuid": uuid,
                "lon": lon,
                "lat": lat,
            })

    done_cvrs.add(cvr)
    # Save progress after each CVR
    save_checkpoint(all_rows, done_cvrs)
    time.sleep(0.3)

# ---------- OUTPUT ----------
pu_coords_df = pd.DataFrame(all_rows)
if not pu_coords_df.empty and "pno" in pu_coords_df.columns:
    pu_coords_df["pno"] = pu_coords_df["pno"].astype("Int64")
    pu_coords_df = pu_coords_df.set_index("pno")

geocoded = pu_coords_df["lon"].notna().sum() if not pu_coords_df.empty else 0
print(f"\nDone! {geocoded}/{len(pu_coords_df)} addresses geocoded across {len(done_cvrs)}/{len(cvr_list)} CVR(s).")
pu_coords_df


--- CVR 16639435 ---
  API fejl: QUOTA_EXCEEDED – Your quota has been exceeded. Reach out if you are certain this is a error. https://cvrapi.dk/contact
Quota exceeded – stopping pipeline.

Done! 0/0 addresses geocoded across 2 CVR(s).


""


# Alternative pipeline using virkdata.dk API
Uses the virkdata.dk API (https://virkdata.dk/api/) with API key auth.
Returns company-level info (name, address, industry, etc.) — no production units endpoint available, so we geocode the company address.

! Needs a subscription to get more than 200 requests a month

In [43]:
# === Alternative pipeline: virkdata.dk API → DAWA geocoding ===

VIRKDATA_API_KEY = "783201g3-2y00-997h-3o86-9233s5116274"  # <-- your virkdata.dk API key
VIRKDATA_URL = "https://virkdata.dk/api/"

CHECKPOINT_PATH_VIRKDATA = os.path.join("..", "data", "interim", "cvr_pipeline_virkdata_checkpoint.json")

def virkdata_lookup(cvr):
    """Fetch company data from virkdata.dk API."""
    resp = requests.get(
        VIRKDATA_URL,
        params={"search": str(cvr)},
        headers={"Authorization": VIRKDATA_API_KEY},
    )
    resp.raise_for_status()
    data = resp.json()
    if "error_code" in data:
        return None
    return data

def save_checkpoint_virkdata(rows, done_cvrs):
    with open(CHECKPOINT_PATH_VIRKDATA, "w", encoding="utf-8") as f:
        json.dump({"rows": rows, "done_cvrs": list(done_cvrs)}, f, ensure_ascii=False)

def load_checkpoint_virkdata():
    if os.path.exists(CHECKPOINT_PATH_VIRKDATA):
        with open(CHECKPOINT_PATH_VIRKDATA, "r", encoding="utf-8") as f:
            cp = json.load(f)
        print(f"Resuming from checkpoint: {len(cp['done_cvrs'])} CVRs done, {len(cp['rows'])} rows.")
        return cp["rows"], set(cp["done_cvrs"])
    return [], set()


assert VIRKDATA_API_KEY, "Set VIRKDATA_API_KEY above before running."

all_rows_vd, done_cvrs_vd = load_checkpoint_virkdata()
remaining_vd = [c for c in cvr_list if c not in done_cvrs_vd]
print(f"Total: {len(cvr_list)} | Already done: {len(done_cvrs_vd)} | Remaining: {len(remaining_vd)}")

for i, cvr in enumerate(remaining_vd):
    print(f"\n[{len(done_cvrs_vd)+i+1}/{len(cvr_list)}] --- CVR {cvr} ---")
    try:
        vdata = virkdata_lookup(cvr)
    except Exception as e:
        print(f"  Fejl ved opslag for CVR {cvr}: {e}")
        continue

    if vdata is None:
        print("  Ikke fundet.")
        continue

    company_name = vdata.get("name", "")
    address = vdata.get("address", "")
    zipcode = vdata.get("zipcode", "")
    city = vdata.get("city", "")
    addr_str = f"{address}, {zipcode} {city}".strip(", ")

    uuid = get_adgangs_uuid(addr_str) if addr_str else None
    lon, lat = get_coords_from_uuid(uuid)
    print(f"  {company_name}: {addr_str}  → UUID={uuid}  ({lon}, {lat})")

    all_rows_vd.append({
        "cvr": vdata.get("vat"),
        "company": company_name,
        "address": address,
        "zipcode": zipcode,
        "city": city,
        "status": vdata.get("status"),
        "startdate": vdata.get("startdate"),
        "enddate": vdata.get("enddate"),
        "employees": vdata.get("employees"),
        "industrycode": vdata.get("industrycode"),
        "industrydesc": vdata.get("industrydesc"),
        "companytype": vdata.get("companytype"),
        "adgangs_uuid": uuid,
        "lon": lon,
        "lat": lat,
    })

    done_cvrs_vd.add(cvr)
    save_checkpoint_virkdata(all_rows_vd, done_cvrs_vd)
    time.sleep(0.2)

# ---------- OUTPUT ----------
pu_coords_df_vd = pd.DataFrame(all_rows_vd)
geocoded = pu_coords_df_vd["lon"].notna().sum() if not pu_coords_df_vd.empty else 0
print(f"\nDone! {geocoded}/{len(pu_coords_df_vd)} geocoded across {len(done_cvrs_vd)}/{len(cvr_list)} CVR(s).")
pu_coords_df_vd

Total: 2 | Already done: 0 | Remaining: 2

[1/2] --- CVR 16639435 ---
  REVIPRO A/S: Greve Strandvej 171, 2670 Greve  → UUID=0a3f5081-2321-32b8-e044-0003ba298018  (12.29144057, 55.57298329)

[3/2] --- CVR 21275913 ---
  FINANS- OG EJENDOMSSELSKABET 'AALYKKE' A/S: Ved Stranden 22 5th, 9000 Aalborg  → UUID=0a3f509c-db86-32b8-e044-0003ba298018  (9.91898415, 57.05098465)

Done! 2/2 geocoded across 2/2 CVR(s).


,cvr,company,address,zipcode,city,status,startdate,enddate,employees,industrycode,industrydesc,companytype,adgangs_uuid,lon,lat
0,16639435,REVIPRO A/S,Greve Strandvej 171,2670,Greve,Normal,2005-04-03,None,0,649990,Anden finansiel formidling i.a.n.,A/S,0a3f5081-2321-32b8-e044-0003ba298018,12.291441,55.572983
1,21275913,FINANS- OG EJENDOMSSELSKABET 'AALYKKE' A/S,Ved Stranden 22 5th,9000,Aalborg,Normal,1990-10-30,None,3,649990,Anden finansiel formidling i.a.n.,A/S,0a3f509c-db86-32b8-e044-0003ba298018,9.918984,57.050985


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Convert to GeoDataFrame (drop rows without coordinates)
df_geo = pu_coords_df.reset_index().dropna(subset=["lon", "lat"]).copy()
geometry = [Point(lon, lat) for lon, lat in zip(df_geo["lon"], df_geo["lat"])]
gdf = gpd.GeoDataFrame(df_geo, geometry=geometry, crs="EPSG:4326")

# Reproject to EPSG:25832
gdf = gdf.to_crs(epsg=25832)

# Save as GeoJSON
out_path = "../data/processed/cvr/CVR_geo.geojson"
gdf.to_file(out_path, driver="GeoJSON")
print(f"Saved {len(gdf)} features to {out_path}")

Saved 63 features to ../data/processed/cvr/CVR_geo.geojson
